<a href="https://colab.research.google.com/github/tomeravgil/Homework5CSCI6170/blob/main/Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1

In [2]:
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(q, k, v):
    # Swap axes handles 3D batch multiplication correctly
    matmul_qk = q @ k.swapaxes(-1, -2)
    dk = k.shape[-1]
    scaled_attention_logits = matmul_qk / np.sqrt(dk)
    attention_weights = softmax(scaled_attention_logits)
    output = attention_weights @ v
    return output, attention_weights

## Part 2

In [9]:
class NumpyRNNEncoder:
    def __init__(self, input_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        self.W_hx = np.random.randn(input_dim, hidden_dim) * 0.1
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.1

    def forward(self, x_seq):
        batch_size, seq_len, _ = x_seq.shape
        h_t = np.zeros((batch_size, self.hidden_dim))

        all_hidden_states = []

        for t in range(seq_len):
            x_t = x_seq[:, t, :]
            h_t = np.tanh(x_t @ self.W_hx + h_t @ self.W_hh)
            all_hidden_states.append(h_t)

        encoder_outputs = np.stack(all_hidden_states, axis=1)
        return encoder_outputs

class LuongAttentionDecoderStep:
    def __init__(self, hidden_dim, vocab_size):
        self.hidden_dim = hidden_dim
        self.W_c = np.random.randn(hidden_dim * 2, hidden_dim) * 0.1
        self.W_vocab = np.random.randn(hidden_dim, vocab_size) * 0.1

    def forward_step(self, decoder_state_t, encoder_outputs):
        """
        decoder_state_t shape: (batch_size, 1, hidden_dim)
        encoder_outputs shape: (batch_size, seq_len, hidden_dim)
        """
        context_vector, attn_weights = scaled_dot_product_attention(
            q=decoder_state_t,
            k=encoder_outputs,
            v=encoder_outputs
        )

        combined_features = np.concatenate([context_vector, decoder_state_t], axis=-1)

        attentional_state = np.tanh(combined_features @ self.W_c)

        logits = attentional_state @ self.W_vocab
        vocab_probs = softmax(logits)

        return vocab_probs, context_vector, attn_weights


if __name__ == "__main__":
    batch_size = 2
    seq_len = 5
    input_dim = 10
    hidden_dim = 16
    vocab_size = 100

    encoder = NumpyRNNEncoder(input_dim, hidden_dim)
    decoder_step = LuongAttentionDecoderStep(hidden_dim, vocab_size)

    mock_input_sequence = np.random.randn(batch_size, seq_len, input_dim)

    encoder_outputs = encoder.forward(mock_input_sequence)
    print(f"Encoder Outputs Shape (K and V): {encoder_outputs.shape}")

    current_decoder_state = np.random.randn(batch_size, 1, hidden_dim)
    print(f"Decoder State Shape (Q):         {current_decoder_state.shape}")

    word_probs, context, weights = decoder_step.forward_step(current_decoder_state, encoder_outputs)

    print(f"\nContext Vector Shape:            {context.shape}")
    print(f"Attention Weights Shape:         {weights.shape}")
    print(f"Vocabulary Probabilities Shape:  {word_probs.shape}")
    print(f"Probabilities sum (should be 1.0): {np.sum(word_probs[0, 0, :]):.2f}")

Encoder Outputs Shape (K and V): (2, 5, 16)
Decoder State Shape (Q):         (2, 1, 16)

Context Vector Shape:            (2, 1, 16)
Attention Weights Shape:         (2, 1, 5)
Vocabulary Probabilities Shape:  (2, 1, 100)
Probabilities sum (should be 1.0): 1.00


## Part 3

In [24]:
import tensorflow as tf
import numpy as np
import urllib.request
import zipfile
import os
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Download NLTK tokenizer data quietly
nltk.download('punkt', quiet=True)

# 1. Download and Prepare the Dataset (Tatoeba English-Spanish)
def load_tatoeba_data(num_samples=10000):
    url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
    file_name = "spa-eng.zip"
    if not os.path.exists(file_name):
        urllib.request.urlretrieve(url, file_name)
        with zipfile.ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall()

    with open("spa-eng/spa.txt", "r", encoding="utf-8") as f:
        lines = f.read().split("\n")

    input_texts = []
    target_texts = []

    for line in lines[:num_samples]:
        if "\t" not in line: continue
        eng, spa = line.split("\t")
        # Add <sos> and <eos> tokens for the Seq2Seq framework
        input_texts.append(f"<sos> {eng.lower()} <eos>")
        target_texts.append(f"<sos> {spa.lower()} <eos>")

    return input_texts, target_texts

input_texts, target_texts = load_tatoeba_data()

# 2. Tokenization and Padding
def tokenize(texts):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
    tokenizer.fit_on_texts(texts)
    tensor = tokenizer.texts_to_sequences(texts)
    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')
    return tensor, tokenizer

input_tensor, inp_lang = tokenize(input_texts)
target_tensor, targ_lang = tokenize(target_texts)

# Train/Test Split (80/20)
split = int(len(input_tensor) * 0.8)
x_train, x_test = input_tensor[:split], input_tensor[split:]
y_train, y_test = target_tensor[:split], target_tensor[split:]

vocab_inp_size = len(inp_lang.word_index) + 1
vocab_tar_size = len(targ_lang.word_index) + 1

In [25]:
class ScaledDotProductAttention(tf.keras.layers.Layer):
    def call(self, q, k, v):
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)
        return output

class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(enc_units, return_sequences=True, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        encoder_output, state_h, state_c = self.lstm(x)
        return encoder_output, [state_h, state_c]

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(dec_units, return_sequences=True, return_state=True)
        self.attention = ScaledDotProductAttention()
        self.concat = tf.keras.layers.Concatenate(axis=-1)
        self.dense = tf.keras.layers.Dense(vocab_size, activation='softmax')

    def call(self, x, hidden_states, encoder_outputs):
        x = self.embedding(x)
        decoder_outputs, state_h, state_c = self.lstm(x, initial_state=hidden_states)
        context_vector = self.attention(decoder_outputs, encoder_outputs, encoder_outputs)
        combined_context = self.concat([context_vector, decoder_outputs])
        output = self.dense(combined_context)
        return output, [state_h, state_c]

class Seq2SeqAttentionModel(tf.keras.Model):
    def __init__(self, encoder, decoder):
        super(Seq2SeqAttentionModel, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        encoder_input, decoder_input = inputs
        encoder_outputs, encoder_states = self.encoder(encoder_input)
        decoder_outputs, _ = self.decoder(decoder_input, encoder_states, encoder_outputs)
        return decoder_outputs

# Initialize
embedding_dim = 128
units = 256
encoder = Encoder(vocab_inp_size, embedding_dim, units)
decoder = Decoder(vocab_tar_size, embedding_dim, units)
model = Seq2SeqAttentionModel(encoder, decoder)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

In [27]:
dec_input_data = y_train[:, :-1]
dec_target_data = y_train[:, 1:]

print("Training Model...")
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model.fit(
    [x_train, dec_input_data],
    dec_target_data,
    batch_size=32,
    epochs=100,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# --- BLEU Score Evaluation ---
def calculate_bleu(model, x_test, y_test, targ_lang, num_samples=None):
    print("\nCalculating BLEU Score...")
    bleu_scores = []
    smoother = SmoothingFunction().method1
    tar_idx2word = {idx: word for word, idx in targ_lang.word_index.items()}
    sos_idx = targ_lang.word_index['<sos>']
    eos_idx = targ_lang.word_index['<eos>']

    n = len(x_test) if num_samples is None else min(len(x_test), num_samples)

    for i in range(n):
        input_seq = x_test[i:i+1]  # (1, seq_len)
        actual_seq = y_test[i]
        actual_words = [tar_idx2word[idx] for idx in actual_seq
                        if idx not in [0, sos_idx, eos_idx]]

        # Run encoder once per sentence
        encoder_outputs, encoder_states = model.encoder(input_seq, training=False)

        target_seq = np.array([[sos_idx]])
        decoded_sentence = []

        for _ in range(20):
            output_tokens, decoder_states = model.decoder(
                target_seq, encoder_states, encoder_outputs, training=False
            )
            sampled_idx = np.argmax(output_tokens[0, -1, :])
            sampled_word = tar_idx2word.get(sampled_idx, '')

            if sampled_word == '<eos>' or sampled_word == '':
                break

            decoded_sentence.append(sampled_word)
            target_seq = np.append(target_seq, [[sampled_idx]], axis=1)
            encoder_states = decoder_states  # update states

        score = sentence_bleu([actual_words], decoded_sentence, smoothing_function=smoother)
        bleu_scores.append(score)

        if i % 100 == 0:
            print(f"  {i}/{n} done...")

    avg_bleu = np.mean(bleu_scores)
    print(f"Average BLEU Score: {avg_bleu:.4f}")
    return avg_bleu

calculate_bleu(model, x_test, y_test, targ_lang, num_samples=len(x_test))

Training Model...
Epoch 1/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 1.1353 - val_loss: 2.4967
Epoch 2/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 1.0068 - val_loss: 2.5882
Epoch 3/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.8979 - val_loss: 2.6165
Epoch 4/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.7993 - val_loss: 2.6734
Epoch 5/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.7099 - val_loss: 2.7184
Epoch 6/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.6294 - val_loss: 2.7775
Epoch 7/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.5549 - val_loss: 2.8390
Epoch 8/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.4870 - val_loss: 2.8617
Epoch 9/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.4284 - val_loss: 2.9115
Epoch 10/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.3776 - val_loss: 2.9569
Epoch 11/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.3360 - val_loss: 3.0079

Calculating BLEU Score

np.float64(0.052897735158474665)

## Results

Intially training the model without early stopping created a bleu score of 0.012. Continuing the training cycle and allowing the model to further minimize its training loss, we increased the BLEU score to 0.0368. A high validation loss as seen shows that the model has reached its capacity of what it can learn from a 2000 sentence dataset without overfitting. To reach a better BLEU score, we can increase the dataset.

## Part 4

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 1. Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=64, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register as a buffer (won't be updated by optimizer)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # Add positional encoding to embeddings
        x = x + self.pe[:, :x.size(1), :]
        return x

# 2. Scaled Dot-Product Attention
def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    # PyTorch transpose swaps specific dimensions (here, the last two)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        # PyTorch uses masked_fill to replace 0s with -infinity before softmax
        scores = scores.masked_fill(mask == 0, -1e9)

    p_attn = F.softmax(scores, dim=-1)
    output = torch.matmul(p_attn, v)
    return output, p_attn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, num_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        # Apply linear layers and split into heads
        q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # REMOVED: mask = mask.unsqueeze(1)
        # The masks are already shaped correctly by our helper functions!

        # Apply attention
        x, attn = scaled_dot_product_attention(q, k, v, mask)

        # Concatenate heads back together
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.w_o(x)

class FeedForward(nn.Module):
    def __init__(self, d_model=64, dff=128, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, dff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dff, d_model)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class EncoderLayer(nn.Module):
    def __init__(self, d_model=64, num_heads=2, dff=128, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, dff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_output)) # Residual 1

        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output)) # Residual 2
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model=64, num_heads=2, dff=128, dropout=0.1):
        super().__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads) # Masked self-attention
        self.mha2 = MultiHeadAttention(d_model, num_heads) # Cross-attention
        self.ffn = FeedForward(d_model, dff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        # 1. Masked self-attention
        attn1 = self.mha1(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn1))

        # 2. Cross-attention
        attn2 = self.mha2(q=x, k=enc_output, v=enc_output, mask=src_mask)
        x = self.norm2(x + self.dropout2(attn2))

        # 3. FFN
        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_output))
        return x

class SimpleTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, num_layers=2, d_model=64, num_heads=2, dff=128, dropout=0.1):
        super().__init__()

        self.src_emb = nn.Embedding(src_vocab_size, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # nn.ModuleList holds lists of submodules properly so PyTorch tracks their gradients
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, dff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, dff, dropout) for _ in range(num_layers)])

        # Final linear layer to output vocabulary probabilities
        self.out = nn.Linear(d_model, tgt_vocab_size)
        self.d_model = d_model

    def forward(self, src, tgt, src_mask, tgt_mask):
        # Encoder Pass
        enc_output = self.src_emb(src) * math.sqrt(self.d_model)
        enc_output = self.pos_encoder(enc_output)

        for layer in self.encoder_layers:
            enc_output = layer(enc_output, src_mask)

        # Decoder Pass
        dec_output = self.tgt_emb(tgt) * math.sqrt(self.d_model)
        dec_output = self.pos_encoder(dec_output)

        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, src_mask, tgt_mask)

        return self.out(dec_output)

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch.utils.data import TensorDataset, DataLoader

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Convert Keras NumPy arrays to PyTorch Tensors
batch_size = 32

# We use torch.long because these are integer indices
train_dataset = TensorDataset(torch.tensor(x_train, dtype=torch.long),
                              torch.tensor(y_train, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Create masks where 1 = Valid Token, 0 = Padding/Masked
def make_src_mask(src):
    # Shape: (batch_size, 1, 1, src_seq_len)
    src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
    return src_mask.to(device)

def make_tgt_mask(tgt):
    # 1. Padding mask
    tgt_pad_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)

    # 2. Look-ahead mask (creates a lower triangular matrix of 1s)
    tgt_len = tgt.size(1)
    tgt_sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), type=torch.uint8)).unsqueeze(0).unsqueeze(0)

    # Combine masks: must not be padding AND must not be future token
    tgt_mask = tgt_pad_mask & tgt_sub_mask
    return tgt_mask.to(device)

Using device: cuda


In [30]:
import copy # Needed to save the model weights
from torch.utils.data import TensorDataset, DataLoader

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Bridge the Keras Data to PyTorch DataLoaders
batch_size = 32

# Training Loader
train_dataset = TensorDataset(torch.tensor(x_train, dtype=torch.long),
                              torch.tensor(y_train, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# NEW: Validation Loader (Using your test split)
val_dataset = TensorDataset(torch.tensor(x_test, dtype=torch.long),
                            torch.tensor(y_test, dtype=torch.long))
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# 3. Masking Functions
def make_src_mask(src):
    src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
    return src_mask.to(device)

def make_tgt_mask(tgt):
    tgt_pad_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)
    tgt_len = tgt.size(1)
    tgt_sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), dtype=torch.bool)).unsqueeze(0).unsqueeze(0).to(device)
    tgt_mask = tgt_pad_mask & tgt_sub_mask
    return tgt_mask

model = SimpleTransformer(src_vocab_size=vocab_inp_size, tgt_vocab_size=vocab_tar_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

epochs = 50
patience = 10
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_weights = None

print("Starting Training with Early Stopping...")

for epoch in range(epochs):
    # --- A. Training Phase ---
    model.train()
    total_train_loss = 0

    for batch_idx, (src, tgt) in enumerate(train_loader):
        src, tgt = src.to(device), tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_expected = tgt[:, 1:]

        src_mask = make_src_mask(src)
        tgt_mask = make_tgt_mask(tgt_input)

        predictions = model(src, tgt_input, src_mask, tgt_mask)
        predictions = predictions.reshape(-1, predictions.size(-1))
        tgt_expected = tgt_expected.reshape(-1)

        loss = criterion(predictions, tgt_expected)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- B. Validation Phase ---
    model.eval() # Turn off dropout for validation
    total_val_loss = 0

    with torch.no_grad(): # Don't track gradients here
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_expected = tgt[:, 1:]

            src_mask = make_src_mask(src)
            tgt_mask = make_tgt_mask(tgt_input)

            predictions = model(src, tgt_input, src_mask, tgt_mask)
            predictions = predictions.reshape(-1, predictions.size(-1))
            tgt_expected = tgt_expected.reshape(-1)

            loss = criterion(predictions, tgt_expected)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # --- C. Early Stopping Check ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        # Deepcopy saves the actual weights, not just a reference!
        best_model_weights = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1
        print(f"  -> No improvement in Val Loss for {epochs_no_improve} epoch(s).")

    if epochs_no_improve >= patience:
        print(f"\nEarly Stopping Triggered! Restoring best weights from epoch {epoch + 1 - patience}...")
        model.load_state_dict(best_model_weights)
        break

# If the loop finishes all 50 epochs without early stopping, still load the best weights
if epochs_no_improve < patience and best_model_weights is not None:
    model.load_state_dict(best_model_weights)

# 6. Evaluation / BLEU Score
def evaluate_bleu_pytorch(model, x_test, y_test, targ_lang, num_samples=50):
    print("\nCalculating BLEU Score using best model weights...")
    model.eval()
    bleu_scores = []
    smoother = SmoothingFunction().method1
    tar_idx2word = {idx: word for word, idx in targ_lang.word_index.items()}

    sos_idx = targ_lang.word_index['<sos>']
    eos_idx = targ_lang.word_index['<eos>']

    with torch.no_grad():
        for i in range(min(len(x_test), num_samples)):
            src = torch.tensor(x_test[i:i+1], dtype=torch.long).to(device)
            src_mask = make_src_mask(src)

            actual_seq = y_test[i]
            actual_words = [tar_idx2word[idx] for idx in actual_seq if idx not in [0, sos_idx, eos_idx]]

            tgt_input = torch.tensor([[sos_idx]], dtype=torch.long).to(device)
            decoded_sentence = []

            for _ in range(20):
                tgt_mask = make_tgt_mask(tgt_input)
                predictions = model(src, tgt_input, src_mask, tgt_mask)
                next_word_idx = predictions[0, -1, :].argmax().item()

                if next_word_idx == eos_idx or next_word_idx == 0:
                    break

                decoded_sentence.append(tar_idx2word.get(next_word_idx, ''))
                next_word_tensor = torch.tensor([[next_word_idx]], dtype=torch.long).to(device)
                tgt_input = torch.cat([tgt_input, next_word_tensor], dim=1)

            score = sentence_bleu([actual_words], decoded_sentence, smoothing_function=smoother)
            bleu_scores.append(score)

    avg_bleu = np.mean(bleu_scores)
    print(f"Average BLEU Score on Test Subset: {avg_bleu:.4f}")
    return avg_bleu

evaluate_bleu_pytorch(model, x_test, y_test, targ_lang)

Using device: cuda
Starting Training with Early Stopping...
Epoch 1/50 - Train Loss: 5.7015 | Val Loss: 5.2765
Epoch 2/50 - Train Loss: 4.3534 | Val Loss: 4.8534
Epoch 3/50 - Train Loss: 3.7220 | Val Loss: 4.5928
Epoch 4/50 - Train Loss: 3.2261 | Val Loss: 4.4778
Epoch 5/50 - Train Loss: 2.7755 | Val Loss: 4.3701
Epoch 6/50 - Train Loss: 2.3765 | Val Loss: 4.4322
  -> No improvement in Val Loss for 1 epoch(s).
Epoch 7/50 - Train Loss: 2.0245 | Val Loss: 4.3829
  -> No improvement in Val Loss for 2 epoch(s).
Epoch 8/50 - Train Loss: 1.7106 | Val Loss: 4.3685
Epoch 9/50 - Train Loss: 1.4502 | Val Loss: 4.4952
  -> No improvement in Val Loss for 1 epoch(s).
Epoch 10/50 - Train Loss: 1.2271 | Val Loss: 4.5389
  -> No improvement in Val Loss for 2 epoch(s).
Epoch 11/50 - Train Loss: 1.0669 | Val Loss: 4.5680
  -> No improvement in Val Loss for 3 epoch(s).
Epoch 12/50 - Train Loss: 0.9401 | Val Loss: 4.8003
  -> No improvement in Val Loss for 4 epoch(s).
Epoch 13/50 - Train Loss: 0.8416 | Va

np.float64(0.09202821030186495)